# 1. SETUP

In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import os

# 2. CONFIG

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32 # Sesuaikan kemampuan GPU

TRAIN_DIR = "data/processed/train"
VAL_DIR   = "data/processed/validation"
TEST_DIR  = "data/processed/test"

EPOCHS = 30 # Bisa diubah 
FINE_TUNE_EPOCHS = 15 # Epoch untuk fine-tuning

INITIAL_LR = 1e-4 # Learning rate di 0.0001
FINE_TUNE_LR = 1e-5 # Learning rate di 0.00001 untuk fine-tuning

# 3. LOAD DATASET

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_gen = ImageDataGenerator(rescale=1./255) # Rescaling data train
val_test_gen = ImageDataGenerator(rescale=1./255) # Rescaling data validasi dan test

# Membuat generator untuk data train
train_data = train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# Membuat generator untuk data validasi
val_data = val_test_gen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# Membuat generator untuk data test
test_data = val_test_gen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False # Untuk evaluasi, agar data urut sesuai label
)

# 4. CALLBACK

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Menerapkan early stopping jika tidak ada peningkatan dalam 7 epoch
callbacks = [
    EarlyStopping(patience=7, restore_best_weights=True),
    ModelCheckpoint("models/best_model.h5", save_best_only=True) # 
]

# 5. FUNCTION EVALUASI

In [ ]:
def evaluate_model(model, name):
    print(f"\n=== {name} ===")
    # Evaluasi model pada data test
    loss, acc = model.evaluate(test_data)
    print("Accuracy:", acc)
    print("Loss:", loss)
    # Prediksi label untuk data test
    y_pred = np.argmax(model.predict(test_data), axis=1)
    y_true = test_data.classes
    # Tampilkan classification report
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=test_data.class_indices.keys()))
    # Tampilkan confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d')
    plt.title(f"Confusion Matrix - {name}")
    plt.show()
    
    return acc

# 6. MODEL

## A. MobileNetV2

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# Membangun MobileNetV2 sebagai base model
def build_mobilenet():
    base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
    base.trainable = False
    
    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    output = layers.Dense(4, activation='softmax')(x)

    model = models.Model(inputs=base.input, outputs=output)
    return model, base

## B. EfficientNetB0

In [ ]:
from tensorflow.keras.applications import EfficientNetB0

def build_efficientnet():
    base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))
    base.trainable = False

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    output = layers.Dense(4, activation='softmax')(x)

    model = models.Model(inputs=base.input, outputs=output)
    return model, base

## C. Hybrid (Feature Fusion)

In [ ]:
def build_fusion():
    input_layer = tf.keras.Input(shape=(224,224,3))

    base1 = MobileNetV2(weights='imagenet', include_top=False, input_tensor=input_layer)
    base2 = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=input_layer)

    base1.trainable = False
    base2.trainable = False

    x1 = layers.GlobalAveragePooling2D()(base1.output)
    x1 = layers.BatchNormalization()(x1)

    x2 = layers.GlobalAveragePooling2D()(base2.output)
    x2 = layers.BatchNormalization()(x2)

    fusion = layers.concatenate([x1, x2])

    x = layers.Dense(256, activation='relu')(fusion)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)

    output = layers.Dense(4, activation='softmax')(x)

    model = models.Model(inputs=input_layer, outputs=output)
    return model, base1, base2

# 7. Training Function (2 Tahap)

In [ ]:
def train_model(model, base_model=None):
    
    # Tahap 1
    model.compile(
        optimizer=tf.keras.optimizers.Adam(INITIAL_LR),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    model.fit(
        train_data,
        validation_data=val_data,
        epochs=EPOCHS,
        callbacks=callbacks
    )

    # Tahap 2 (Fine-tuning)
    if base_model is not None:
        for layer in base_model.layers[-30:]:
            layer.trainable = True

    model.compile(
        optimizer=tf.keras.optimizers.Adam(FINE_TUNE_LR),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    model.fit(
        train_data,
        validation_data=val_data,
        epochs=FINE_TUNE_EPOCHS,
        callbacks=callbacks
    )

    return model

# 8. Train & Evaluasi Model

## A. MobileNetV2

In [ ]:
model_m, base_m = build_mobilenet()
model_m = train_model(model_m, base_m)
acc_m = evaluate_model(model_m, "MobileNetV2")

## B. EfficientNetB0

In [ ]:
model_e, base_e = build_efficientnet()
model_e = train_model(model_e, base_e)
acc_e = evaluate_model(model_e, "EfficientNetB0")

## C. Hybrid Model

In [ ]:
model_f, base1, base2 = build_fusion()

# fine-tune dua backbone
model_f.compile(
    optimizer=tf.keras.optimizers.Adam(INITIAL_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_f.fit(train_data, validation_data=val_data, epochs=EPOCHS, callbacks=callbacks)

for layer in base1.layers[-30:]:
    layer.trainable = True
for layer in base2.layers[-30:]:
    layer.trainable = True

model_f.compile(
    optimizer=tf.keras.optimizers.Adam(FINE_TUNE_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_f.fit(train_data, validation_data=val_data, epochs=FINE_TUNE_EPOCHS, callbacks=callbacks)

acc_f = evaluate_model(model_f, "Fusion Model")

# 9. Perbandingan Hasil

In [ ]:
print("\n=== HASIL AKHIR ===")
print(f"MobileNetV2   : {acc_m}")
print(f"EfficientNet  : {acc_e}")
print(f"Fusion Model  : {acc_f}")